In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from smolagents import CodeAgent, DuckDuckGoSearchTool, HfApiModel, FinalAnswerTool, GoogleSearchTool, VisitWebpageTool
import random
import yaml
import PIL
import numpy as np

import torch

from src.agents.utils import FinalAnswerTool
from src.agents.tools.reid import ReIDTool
from src.agents.tools.inpainting import InpaintingTool
from src.agents.tools.segmentation import SemanticSegmentationTool
# from src.agents.tools.transmission import EncodingTool, DecodingTool
from src.agents.tools.comm import EncodingTool, DecodingTool

from src.agents.tools.refiner.kitti_tracking import KittiDataset

from matplotlib import pyplot as plt

c:\Users\ngoak\anaconda3\envs\agentic\lib\site-packages\torchreid\reid\metrics\rank.py:11: UserWarning: Cython evaluation (very fast so highly recommended) is unavailable, now use python evaluation.
  warnings.warn(


In [3]:
root_dir = "E:/KittiTracking"
n_steps, n_pred_steps = 16, 3

train_ds = KittiDataset(root_dir, "train", n_steps, n_pred_steps)
for sample in train_ds:
	frames, _ = sample
	break

In [69]:
sem_segm_tool = SemanticSegmentationTool(model_name="CIDAS/clipseg-rd64-refined")
encoder_tool = EncodingTool("https://cdn.openai.com/dall-e/encoder.pkl")
deocder_tool = DecodingTool("https://cdn.openai.com/dall-e/decoder.pkl")

In [70]:
# 1/ extract application requirement: description, modalities, and object classes
app_des = "autonomous driving"
obj_cls = "vehicles, cars, pedestrians, cyclists, and road areas".split(", ")

# 2/ extract mask for each modality
# 3/ 
for frame in frames:
    frame = frame.resize((640, 480))

    sem_mask = sem_segm_tool(frame=frame, prompt=obj_cls)
    codewords = encoder_tool(sem_mask)
    # comm_mask, codewords = output_dict["comm_mask"], output_dict["codewords"]
    rec_sem_mask = deocder_tool(codewords, False)
    break

In [56]:
codewords.shape

(1, 60, 80)

In [64]:
flatten_codewords = [
    codewords[0][i][j]
    for i in range(len(codewords[0]))
    for j in range(len(codewords[0, 0]))
    if comm_mask[i][j]
]
len(flatten_codewords) / (60*80)

0.4610416666666667